# Ch 19 · Constraints as Convex Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/19-constraints-as-optimization.ipynb)

Companion notebook for the chapter page `chapters/19-constraints-as-optimization.html` ([page](../chapters/19-constraints-as-optimization.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/19-constraints-as-optimization/`). Most examples use one world per case, so they are small on a GPU too; examples with an `ON_GPU` budget grow on a GPU. MJWarp computes in float32, so expect differences of about 1e-4 to 1e-6 against MuJoCo C (float64). The figure-data flags (`--dump-json`) are not needed here.

Order: the cells follow the chapter page: Example 19.1, the NumPy math toolkit (Example 19.12), then Examples 19.2, 19.3, 19.9, 19.11, 19.10 and 19.4-19.8. The centerpiece is Example 19.3, which rebuilds the cost from MJWarp's arrays and proves that `qacc` is its minimizer.

## 1. The 1-D toy: one ball, one row
The ball's acceleration minimizes `0.5 m (a - a0)^2 + 0.5 D (a - aref)^2` (second term only while `a < aref`). The closed form is `a* = (m a0 + D aref)/(m + D)`. `k, b, imp` come from solref/solimp as in `constraint.py::_contact_kbimp`, then `D = imp/((1-imp) invweight)` and `aref = -b v - k imp pos` (`_efc_row`). Each world is one (penetration, velocity) case.

In [ ]:
# Example 19.1 - ex1_toy_1d.py
"""Example 19.1 - the 1-D ball-on-floor toy: one soft contact row, solved by hand and by MJWarp.

A ball (mass 1) is pressed into the floor by depth p and moves with vertical velocity v. Its
acceleration a minimizes 0.5*m*(a - a0)^2 + 0.5*D*(a - aref)^2 (the second term only while
a < aref, i.e. while the row is active). We compute k, b, imp, D and aref from solref/solimp
exactly as constraint.py does, take the closed-form optimum, and compare with MJWarp: one world
per (p, v) pair. Floor and ball use condim="1" (a contact takes the larger condim of its two
geoms), so the contact has exactly one row: CONTACT_FRICTIONLESS.
Run: python ex1_toy_1d.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" condim="1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" condim="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

CASES = [(0.0002, 0.0), (0.0005, 0.0), (0.001, 0.0), (0.002, 0.0), (0.005, 0.0),
         (0.002, -0.5), (0.002, 0.5), (-0.001, 0.0)]          # (penetration p [m], v [m/s])
NWORLD = len(CASES)                    # one world per case (same tiny budget on GPU and CPU)


def kbimp(timestep, solref, solimp, pos):
    """NumPy copy of constraint.py::_contact_kbimp for positive (standard) solref."""
    timeconst, dampratio = solref
    dmin, dmax, width, mid, power = solimp
    timeconst = max(timeconst, 2.0 * timestep)                 # refsafe
    k = 1.0 / (dmax**2 * timeconst**2 * dampratio**2)
    b = 2.0 / (dmax * timeconst)
    x = abs(pos) / width
    if x >= 1.0:
        imp = dmax
    elif x <= mid:
        imp = dmin + (dmax - dmin) * x**power / mid ** (power - 1)
    else:
        imp = dmin + (dmax - dmin) * (1 - (1 - x) ** power / (1 - mid) ** (power - 1))
    return k, b, imp


mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=2, njmax=4)

qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qvel = np.zeros((NWORLD, mjm.nv))
for w, (p, v) in enumerate(CASES):
    qpos[w, 2] = 0.1 - p                # ball centre: radius minus penetration
    qvel[w, 2] = v
d.qpos.assign(qpos)
d.qvel.assign(qvel)
mjw.forward(m, d)

mass = 1.0
a0 = mjm.opt.gravity[2]                 # unconstrained acceleration (qacc_smooth, z)
h = mjm.opt.timestep
solref, solimp = mjm.geom_solref[1], mjm.geom_solimp[1]
invweight = float(m.body_invweight0.numpy()[0, 1, 0])   # ball's translational invweight0
print(f"solref {solref}  solimp {solimp}  invweight0 {invweight:.4f}  a0 {a0:.2f}")

nefc = d.nefc.numpy()
D_w, aref_w = d.efc.D.numpy()[:, 0], d.efc.aref.numpy()[:, 0]
f_w, a_w = d.efc.force.numpy()[:, 0], d.qacc.numpy()[:, 2]
print("   p[mm]  v     imp     D       aref      a* toy   a* mjw   f toy    f mjw")
for w, (p, v) in enumerate(CASES):
    pos = -p                            # dist - margin (negative = penetrating)
    if pos >= 0:                        # the row does not exist: no contact force at all
        print(f"  {p*1e3:5.2f} {v:5.2f}   inactive: nefc = {nefc[w]}, a* = a0 = {a_w[w]:.3f}")
        continue
    k, b, imp = kbimp(h, solref, solimp, pos)
    D = 1.0 / (invweight * (1.0 - imp) / imp)    # constraint.py::_efc_D
    aref = -k * imp * pos - b * v                # constraint.py::_efc_row (continuous time)
    if a0 < aref:                                # row active at the optimum
        a_star = (mass * a0 + D * aref) / (mass + D)
    else:
        a_star = a0
    f = -D * (a_star - aref) if a_star < aref else 0.0
    print(f"  {p*1e3:5.2f} {v:5.2f}  {imp:.4f} {D:7.2f} {aref:9.3f}  {a_star:7.3f}  {a_w[w]:7.3f}"
          f"  {f:7.3f}  {f_w[w]:7.3f}")
print("rows per world (nefc):", nefc, "| efc.type world 0:",
      mujoco.mjtConstraint(int(d.efc.type.numpy()[0, 0])).name)
print("Newton's law at the optimum, m*(a* - a0) = f, world 3:",
      round(mass * (a_w[3] - a0), 3), "=", round(float(f_w[3]), 3))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Math toolkit (chapter section 2)
A 2-D SPD quadratic minimized by gradient descent (fixed step), Newton (1 step) and conjugate gradient (2 steps), then a 1-D piecewise quadratic with three kinks, minimized exactly by Newton steps that jump from piece to piece. This is the vocabulary of the solvers in Ch 20: `SolverType.CG`, `SolverType.NEWTON` and the exact line search.

In [ ]:
# Example 19.12 - ex12_math_toolkit.py
"""Example 19.12 - the math toolkit the solver needs, in plain NumPy (no MJWarp).

Part 1: minimize the 2-D SPD quadratic f(x) = 0.5 x^T A x - b^T x three ways:
gradient descent with a fixed step, one Newton step, and two conjugate-gradient steps.
Part 2: a 1-D piecewise quadratic with 3 kinks (the shape of the solver's cost along a search
ray): find its exact minimizer by checking each smooth piece, then by Newton steps that jump
from piece to piece.
Run: python ex12_math_toolkit.py [--dump-json PATH]
  --dump-json PATH writes the iterates and the piecewise curve for the toolkit figure (Rule D).
"""
import argparse
import json

import numpy as np

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

# ---------------- Part 1: a 2-D SPD quadratic ----------------
A = np.array([[4.0, 1.4], [1.4, 1.0]])           # symmetric positive definite
b = np.array([1.0, 2.0])
x0 = np.array([-2.0, -1.0])
x_star = np.linalg.solve(A, b)                     # the unique minimizer: A x = b
lam = np.linalg.eigvalsh(A)
print("eigenvalues of A:", lam, "-> SPD:", bool(lam.min() > 0),
      f"| condition number {lam.max() / lam.min():.1f}")
print("Cholesky factor L (A = L L^T):\n", np.linalg.cholesky(A))
print("x* = A^-1 b =", x_star)


def f(x):
    return 0.5 * x @ A @ x - b @ x


def grad(x):
    return A @ x - b


# gradient descent, fixed step 1/lambda_max (safe for any SPD A)
step = 1.0 / lam.max()
gd = [x0.copy()]
x = x0.copy()
for _ in range(40):
    x = x - step * grad(x)
    gd.append(x.copy())

# Newton: x1 = x0 - H^-1 grad, with H = A: exact in one step for a quadratic
newton = [x0.copy(), x0 - np.linalg.solve(A, grad(x0))]

# conjugate gradient (exact linesearch along A-conjugate directions): exact in n = 2 steps
cg = [x0.copy()]
x = x0.copy()
r = -grad(x)
p = r.copy()
for _ in range(2):
    alpha = (r @ r) / (p @ A @ p)                  # exact 1-D minimizer along p
    x = x + alpha * p
    r_new = r - alpha * (A @ p)
    beta = (r_new @ r_new) / (r @ r)               # Fletcher-Reeves (= every CG on a quadratic)
    p = r_new + beta * p
    r = r_new
    cg.append(x.copy())

for name, path in (("gradient descent", gd), ("Newton", newton), ("CG", cg)):
    errs = [np.linalg.norm(p - x_star) for p in path]
    shown = ", ".join(f"{e:.2e}" for e in errs[:4])
    print(f"{name:17s} {len(path) - 1:2d} steps | |x_k - x*| for k=0..3: {shown}"
          f" | final {errs[-1]:.2e}")
print("GD iterates 1..3:", np.array(gd[1:4]).round(4).tolist())
print(f"f(x*) = {f(x_star):.4f}; every local minimum of a convex f is this global one")

# ---------------- Part 2: a 1-D piecewise quadratic ----------------
# cost(alpha) = 0.5*a*alpha^2 + g*alpha  (the Gauss term along a ray)
#             + sum_i 0.5*D_i*(r_i + j_i*alpha)^2  only where r_i + j_i*alpha < 0 (row active)
a, g = 1.0, -1.0
rows = [(-0.5, 1.0, 4.0),      # (r_i, j_i, D_i): active for alpha < 0.5
        (0.6, -1.0, 3.0),      # active for alpha > 0.6
        (1.5, -1.0, 6.0)]      # active for alpha > 1.5
kinks = sorted(-r / j for r, j, _ in rows)
print("kinks (row switches) at alpha =", [round(k, 3) for k in kinks])


def pq(alpha):
    """Return (cost, slope, curvature) at alpha; the cost is C1, the curvature jumps at kinks."""
    c, s, h = 0.5 * a * alpha**2 + g * alpha, a * alpha + g, a
    for r, j, D in rows:
        x = r + j * alpha
        if x < 0:
            c += 0.5 * D * x * x
            s += D * j * x
            h += D * j * j
    return c, s, h


# exact minimizer: on each smooth piece the cost is one quadratic; take its stationary point
# if it lies inside that piece (convexity guarantees exactly one such point)
edges = [-np.inf] + kinks + [np.inf]
exact = None
for lo, hi in zip(edges[:-1], edges[1:]):
    mid = 0.5 * (max(lo, -10) + min(hi, 10))
    _, s_mid, h_mid = pq(mid)
    cand = mid - s_mid / h_mid                      # Newton is exact inside one piece
    if lo - 1e-12 <= cand <= hi + 1e-12:
        exact = cand
print(f"exact minimizer alpha* = {exact:.6f}, cost {pq(exact)[0]:.6f}, slope {pq(exact)[1]:.1e}")

# Newton steps on the piecewise quadratic: each step is exact for the current piece
alpha = 0.0
newton_1d = [alpha]
for _ in range(8):
    _, s, h = pq(alpha)
    if abs(s) < 1e-12:
        break
    alpha = alpha - s / h
    newton_1d.append(alpha)
print("1-D Newton iterates:", [round(v, 6) for v in newton_1d], f"({len(newton_1d) - 1} steps)")

if args.dump_json:
    grid = np.linspace(-0.5, 2.5, 121)
    curve = [[round(float(t), 4), round(float(pq(t)[0]), 4)] for t in grid]
    out = {"A": A.tolist(), "b": b.tolist(), "x0": x0.tolist(), "xstar": x_star.round(5).tolist(),
           "gd": np.array(gd).round(4).tolist(), "newton": np.array(newton).round(4).tolist(),
           "cg": np.array(cg).round(4).tolist(),
           "pq": {"a": a, "g": g, "rows": rows, "kinks": [round(k, 4) for k in kinks],
                  "exact": round(float(exact), 5), "newton": [round(v, 5) for v in newton_1d],
                  "curve": curve}}
    with open(args.dump_json, "w") as fh:
        json.dump(out, fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 4. The efc arrays
Rows come out per world in the order equality | friction | limit | contact. One row of every category, compared with MuJoCo C. Note the limit row: violated (`pos < 0`) but SATISFIED, because activity is decided at the optimum.

In [ ]:
# Example 19.2 - ex2_inspect_efc.py
"""Example 19.2 - read the efc arrays: one row of every category, side by side with MuJoCo C.

The scene has a box resting on a plane (4 contacts, condim 3, pyramidal: 4 rows each), a hinge
with frictionloss that starts past its range (one friction row and one limit row) and a second
hinge tied to the first by a joint equality (one equality row). Rows come out per world in the
fixed category order  equality | friction | limit | contact.
Run: python ex2_inspect_efc.py [--dump-json PATH]
  --dump-json PATH writes the row layout of both worlds for the efc strip (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True, linewidth=100)

SCENE = """
<mujoco model="efc_tour">
  <option timestep="0.005" cone="pyramidal"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box" pos="0 0 0.099">
      <freejoint/>
      <geom type="box" size=".1 .1 .1" condim="3" friction="0.8"/>
    </body>
    <body name="arm" pos="1 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0" range="-30 30" frictionloss="0.05"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" contype="0" conaffinity="0"/>
    </body>
    <body name="arm2" pos="1.5 0 1">
      <joint name="hinge2" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.03" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <equality><joint joint1="hinge2" joint2="hinge" polycoef="0 1 0 0 0"/></equality>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 2
TYPE = {int(v): k[len("mjCNSTR_"):] for k, v in mujoco.mjtConstraint.__members__.items()}
STATE = {int(v): k[len("mjCNSTRSTATE_"):] for k, v in mujoco.mjtConstraintState.__members__.items()}

mjm = mujoco.MjModel.from_xml_string(SCENE)
mjd = mujoco.MjData(mjm)
hinge_q = mjm.jnt_qposadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_JOINT, "hinge")]
mjd.qpos[hinge_q] = np.radians(35.0)          # 5 degrees past the +30 degree limit
mujoco.mj_forward(mjm, mjd)

m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=8, njmax=32)
mjw.forward(m, d)

print(f"nv {mjm.nv}  nv_pad {m.nv_pad}  is_sparse {m.is_sparse}  njmax {d.njmax}"
      f"  njmax_pad {d.njmax_pad}  naconmax {d.naconmax}")
print("efc.J", d.efc.J.shape, "| efc.D", d.efc.D.shape, "| efc.force", d.efc.force.shape,
      "| contact.efc_address", d.contact.efc_address.shape)
print("ne", d.ne.numpy(), "nf", d.nf.numpy(), "nl", d.nl.numpy(), "nefc", d.nefc.numpy(),
      "| nacon (all worlds)", d.nacon.numpy())
print(f"MuJoCo C: ne {mjd.ne} nf {mjd.nf} nl {mjd.nl} nefc {mjd.nefc} ncon {mjd.ncon}")

n = int(d.nefc.numpy()[0])
typ = d.efc.type.numpy()[0, :n]
st = d.efc.state.numpy()[0, :n]
idx = d.efc.id.numpy()[0, :n]
force = d.efc.force.numpy()[0, :n]
pos = d.efc.pos.numpy()[0, :n]
D = d.efc.D.numpy()[0, :n]
aref = d.efc.aref.numpy()[0, :n]
print("row  type                 id  state       pos      D        aref     force   C force")
for r in range(n):
    print(f"{r:3d}  {TYPE[typ[r]]:19s} {idx[r]:3d}  {STATE[st[r]]:10s} {pos[r]:7.4f} {D[r]:8.2f}"
          f" {aref[r]:9.3f} {force[r]:8.4f} {mjd.efc_force[r]:8.4f}")
print("max |force - C force| =", f"{np.abs(force - mjd.efc_force).max():.1e}",
      "(float32 vs float64)")
nc = int(d.nacon.numpy()[0])
w0 = d.contact.worldid.numpy()[:nc] == 0
print("contact.efc_address (world 0 contacts):\n", d.contact.efc_address.numpy()[:nc][w0])
print("solver_niter", d.solver_niter.numpy(), "| C", mjd.solver_niter[0])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    blocks = []
    for w in range(NWORLD):
        ne, nf, nl = int(d.ne.numpy()[w]), int(d.nf.numpy()[w]), int(d.nl.numpy()[w])
        row = [["eq", ne], ["friction", nf], ["limit", nl]]
        ca = d.contact.efc_address.numpy()[:nc]
        cw = d.contact.worldid.numpy()[:nc]
        cons = sorted((int(ca[i, 0]), i) for i in range(nc) if cw[i] == w)
        row += [["contact", 4, f"c{i}"] for _, i in cons]
        blocks.append(row)
    out = {"njmax": int(d.njmax), "rows": blocks,
           "types": [TYPE[t] for t in typ], "states": [STATE[s] for s in st],
           "force": [round(float(v), 4) for v in force]}
    with open(args.dump_json, "w") as fh:
        json.dump(out, fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. Verify the optimum yourself (centerpiece)
Rebuild `cost(a) = 0.5 (a - qacc_smooth)^T M (a - qacc_smooth) + sum_active 0.5 D (J a - aref)^2` in NumPy, recompute the forces, check `M (qacc - qacc_smooth) = J^T force`, and check that random perturbations of `qacc` never lower the cost.

In [ ]:
# Example 19.3 - ex3_rebuild_cost.py
"""Example 19.3 - rebuild the primal cost in NumPy from MJWarp's arrays and verify the optimum.

A box tilted by 10 degrees rests on one edge (2 contacts, condim 3, pyramidal: 8 rows) and
slides with a different initial speed in each world. For every world we read efc.J, efc.D,
efc.aref, qacc_smooth and the dense M, rebuild
    cost(a) = 0.5 (a - qacc_smooth)^T M (a - qacc_smooth) + sum_active 0.5 D (J a - aref)^2,
recompute the forces from the cost, check M (qacc - qacc_smooth) = J^T force, and check that
random perturbations of qacc never lower the cost (qacc is the minimizer).
Run: python ex3_rebuild_cost.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True, linewidth=100)

TILTED_BOX = """
<mujoco model="tilted_box">
  <option timestep="0.002" cone="pyramidal"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box" pos="0 0 .098" euler="0 10 0">
      <freejoint/>
      <geom type="box" size=".1 .1 .1" friction="0.5"/>
    </body>
  </worldbody>
</mujoco>
"""

SPEEDS = [0.0, 0.3, 1.0, 3.0]                  # initial x velocity per world [m/s]
NWORLD = len(SPEEDS)

mjm = mujoco.MjModel.from_xml_string(TILTED_BOX)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
qvel = np.zeros((NWORLD, mjm.nv))
qvel[:, 0] = SPEEDS
d.qvel.assign(qvel)
mjw.forward(m, d)

nv = mjm.nv
mjd = mujoco.MjData(mjm)
rng = np.random.default_rng(0)
for w in range(NWORLD):
    n = int(d.nefc.numpy()[w])
    J = d.efc.J.numpy()[w, :n, :nv]               # dense: (njmax_pad, nv_pad), keep the used part
    D = d.efc.D.numpy()[w, :n]
    aref = d.efc.aref.numpy()[w, :n]
    qacc = d.qacc.numpy()[w].astype(np.float64)
    qs = d.qacc_smooth.numpy()[w].astype(np.float64)
    mjw.get_data_into(mjd, mjm, d, world_id=w)    # copy world w (incl. qM) into an MjData
    M = np.zeros((nv, nv))
    mujoco.mj_fullM(mjm, mjd, M)

    def cost(a):
        jar = J @ a - aref
        return 0.5 * (a - qs) @ M @ (a - qs) + np.where(jar < 0, 0.5 * D * jar**2, 0.0).sum()

    jar = J @ qacc - aref
    f_np = np.where(jar < 0, -D * jar, 0.0)        # force = -d(row cost)/d(jar)
    f_mjw = d.efc.force.numpy()[w, :n]
    resid = np.abs(M @ (qacc - qs) - J.T @ f_np).max()
    # Ma and qfrc_smooth come from MJWarp itself: the same balance without C MuJoCo
    resid_mjw = np.abs(d.efc.Ma.numpy()[w] - d.qfrc_smooth.numpy()[w]
                       - J.T @ f_mjw).max()
    c0 = cost(qacc)
    worse = min(cost(qacc + 1e-3 * rng.standard_normal(nv)) - c0 for _ in range(200))
    print(f"world {w}: v0 = {SPEEDS[w]:.1f} m/s  nefc {n}  cost {c0:10.4f}")
    print("  state       ", d.efc.state.numpy()[w, :n])
    print("  force (cost)", f_np.round(3))
    print("  efc.force   ", f_mjw.round(3))
    print(f"  max |M(qacc-qacc_smooth) - J^T f| = {resid:.1e}   (Ma - qfrc_smooth - J^T f:"
          f" {resid_mjw:.1e})")
    print(f"  smallest cost change under 200 random 1e-3 perturbations: {worse:+.2e}"
          f" (relative {worse / abs(c0):+.1e})")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 6. Per-world eq_active seen through the efc strip
Recall Ch 8 ex6 (same toggle). New here: the weld block always takes rows 0-5, and where it is switched off every contact's `contact.efc_address` shifts down by 6.

In [ ]:
# Example 19.9 - ex9_eq_active_worlds.py
"""Example 19.9 - per-world eq_active seen through the efc strip: where each row block lands.

Ch 8 ex6 toggled the MOCAP_TARGET weld per world and counted d.ne. Here the welded box rests on
the floor, so every world also has 4 contacts (condim 3, pyramidal: 4 rows each). The weld
block (6 rows) always comes first, because make_constraint launches the equality kernels before
the contact kernels. In worlds where the weld is off, the block disappears and every contact's
contact.efc_address shifts down by 6. We print, per world, the row blocks and the first row of
each contact.
Run: python ex9_eq_active_worlds.py [--dump-json PATH]
  --dump-json PATH writes the per-world row blocks for the efc strip figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

MOCAP_TARGET = """
<mujoco model="mocap_target">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="target" mocap="true" pos="0 0 0.0495">
      <geom type="box" size=".05 .05 .05" contype="0" conaffinity="0" rgba="0 1 0 .3"/>
    </body>
    <body name="box" pos="0 0 0.0495">
      <freejoint/>
      <geom name="box" type="box" size=".05 .05 .05" mass="0.5" rgba=".3 .3 1 1"/>
    </body>
  </worldbody>
  <equality><weld body1="box" body2="target" solref="0.02 1"/></equality>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
SHOW = 4                                    # worlds printed

mjm = mujoco.MjModel.from_xml_string(MOCAP_TARGET)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)

active = np.ones((NWORLD, mjm.neq), dtype=bool)
active[1::2, 0] = False                     # odd worlds: weld off (recall Ch 8 ex6)
d.eq_active.assign(active)
mjw.forward(m, d)

TYPE = {int(v): k[len("mjCNSTR_"):] for k, v in mujoco.mjtConstraint.__members__.items()}
ne, nefc = d.ne.numpy(), d.nefc.numpy()
nacon = int(d.nacon.numpy()[0])
cw = d.contact.worldid.numpy()[:nacon]
ca = d.contact.efc_address.numpy()[:nacon]
etype = d.efc.type.numpy()

print("eq_active[:, 0]:", active[:SHOW, 0].astype(int), "| ne:", ne[:SHOW], "| nefc:", nefc[:SHOW])
print("world  eq_active  row blocks (kind[first:last])           contacts: id -> first row")
strip = []
for w in range(SHOW):
    blocks, r = [], 0
    if ne[w]:
        blocks.append(["eq", int(ne[w]), "weld"])
        r = int(ne[w])
    cons = sorted((int(ca[c, 0]), c) for c in range(nacon) if cw[c] == w)
    for first, c in cons:
        blocks.append(["contact", 4, f"c{c}"])
    txt = " ".join(f"{b[0]}[{s}:{s + b[1] - 1}]" for b, s in
                   zip(blocks, np.cumsum([0] + [b[1] for b in blocks[:-1]])))
    links = " ".join(f"c{c}->{first}" for first, c in cons)
    print(f"  {w}       {int(active[w, 0])}      {txt:40s} {links}")
    strip.append(blocks)
row_types = sorted({TYPE[int(t)] for t in etype[0, :nefc[0]]})
print("row types in world 0:", row_types)
first_rows = {w: sorted(int(ca[c, 0]) for c in range(nacon) if cw[c] == w) for w in range(SHOW)}
print("first contact row, weld on vs off:", first_rows[0][0], "vs", first_rows[1][0],
      f"(shift = ne = {int(ne[0])})")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as fh:
        json.dump({"njmax": int(d.njmax), "eq_active": active[:SHOW, 0].astype(int).tolist(),
                   "rows": strip, "first_rows": [first_rows[w] for w in range(SHOW)]},
                  fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. make_constraint and solve by hand
Both are public. Rebuilding the rows and re-solving after `forward` gives a bitwise-identical `qacc`.

In [ ]:
# Example 19.11 - ex11_by_hand.py
"""Example 19.11 - run the constraint stages by hand: make_constraint, then solve.

forward() already calls collision -> make_constraint -> ... -> solve. Here we let three stacked
boxes settle, run forward once and keep qacc, then call mjw.make_constraint(m, d) and
mjw.solve(m, d) ourselves. make_constraint rebuilds the efc rows from the current contacts and
kinematics; solve needs qacc_smooth, qfrc_smooth and the factorized M that forward left in d.
The result must be identical. Then we disable warmstart and solve again from qacc_smooth: the
same answer within float32 noise. Newton still needs one iteration: at qacc_smooth (free fall)
every contact row is active, the optimum has the same active set, and one Newton step on a
single quadratic piece is exact (section 2 of the chapter).
Run: python ex11_by_hand.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)


def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 1000 if ON_GPU else 40

mjm = mujoco.MjModel.from_xml_string(box_stack(3))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
for _ in range(NSTEP):
    mjw.step(m, d)

mjw.forward(m, d)
qacc_fwd = d.qacc.numpy().copy()
force_fwd = d.efc.force.numpy().copy()
niter_fwd = d.solver_niter.numpy().copy()
print("nv", mjm.nv, "| nefc", d.nefc.numpy()[:2], "| nacon", d.nacon.numpy(),
      "| solver_niter after forward", niter_fwd[:2])

mjw.make_constraint(m, d)                       # rows from d.contact and the kinematics
mjw.solve(m, d)                                 # uses qacc_smooth, qfrc_smooth, qLD from forward
print("max |qacc(by hand) - qacc(forward)| =", float(np.abs(d.qacc.numpy() - qacc_fwd).max()))
print("max |efc.force difference|         =", float(np.abs(d.efc.force.numpy() - force_fwd).max()))

m.opt.disableflags |= mjw.DisableBit.WARMSTART  # start from qacc_smooth instead
mjw.solve(m, d)
print("no warmstart: solver_niter", d.solver_niter.numpy()[:2],
      "| max |qacc - qacc(forward)| =", f"{np.abs(d.qacc.numpy() - qacc_fwd).max():.1e}")
m.opt.disableflags &= ~mjw.DisableBit.WARMSTART
print("box z:", d.qpos.numpy()[0, [2, 9, 16]])
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 7. Dense vs sparse J
`jacobian="auto"` is sparse for `nv > 32`. Sparse `efc.J` is a value ribbon `(nworld, 1, njmax_nnz)` indexed by `J_rowadr`, `J_rownnz`, `J_colind` (columns in descending order). Dense is `(nworld, njmax_pad, nv_pad)`.

In [ ]:
# Example 19.10 - ex10_sparse_j.py
"""Example 19.10 - the sparse (CSR) Jacobian vs the dense one, on 6 free boxes (nv 36).

Three stacks of two boxes: each lower box touches the floor (rows touch its 6 dofs) and each
upper box touches the lower one (rows touch 12 dofs). With jacobian="auto", nv 36 > 32 makes
the model sparse: efc.J is one value ribbon of length njmax_nnz per world, indexed by J_rowadr,
J_rownnz and J_colind. Forcing jacobian="dense" stores a (njmax_pad, nv_pad) block per world.
We print the CSR arrays, the descending column order, the shapes and the bytes of each layout.
Run: python ex10_sparse_j.py [--dump-json PATH]
  --dump-json PATH writes the CSR arrays of world 0 for the J-layout figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True, linewidth=110)


def stacks(nstack=3, size=0.1):
    bodies = ""
    for s in range(nstack):
        x = 0.5 * s
        for k in range(2):
            z = size + 2 * size * k - 0.0005 * (k + 1)       # 0.5 mm overlap at each interface
            bodies += (f'<body name="b{s}{k}" pos="{x} 0 {z:.4f}"><freejoint/>'
                       f'<geom type="box" size="{size} {size} {size}"/></body>')
    return (f'<mujoco model="stacks"><option timestep="0.005"/><worldbody>'
            f'<geom name="floor" type="plane" size="5 5 .1"/>{bodies}</worldbody></mujoco>')


parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 2
NJMAX = 128

mjm = mujoco.MjModel.from_xml_string(stacks())
results = {}
for mode in ("auto", "dense"):
    mjm.opt.jacobian = mujoco.mjtJacobian.mjJAC_AUTO if mode == "auto" else \
        mujoco.mjtJacobian.mjJAC_DENSE
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=NJMAX)
    mjw.forward(m, d)
    n = int(d.nefc.numpy()[0])
    # float32 values and int32 indices: 4 bytes per entry (the index arrays are empty if dense)
    nbytes = 4 * sum(a.size for a in (d.efc.J, d.efc.J_colind, d.efc.J_rownnz, d.efc.J_rowadr))
    print(f"{mode:5s}: nv {mjm.nv}  is_sparse {m.is_sparse}  nv_pad {m.nv_pad}  njmax {d.njmax}"
          f"  njmax_pad {d.njmax_pad}  njmax_nnz {d.njmax_nnz}  nefc {n}")
    print(f"       efc.J {d.efc.J.shape}  J_rownnz {d.efc.J_rownnz.shape}"
          f"  J_colind {d.efc.J_colind.shape}  -> {nbytes / 1024:.1f} KiB for {NWORLD} worlds")
    results[mode] = dict(m=m, d=d, n=n, nbytes=nbytes)

ds, dd = results["auto"]["d"], results["dense"]["d"]
n = results["auto"]["n"]
rownnz = ds.efc.J_rownnz.numpy()[0, :n]
rowadr = ds.efc.J_rowadr.numpy()[0, :n]
colind = ds.efc.J_colind.numpy()[0, 0]
vals = ds.efc.J.numpy()[0, 0]
print("J_rownnz (world 0):", rownnz)
print("J_rowadr first 12 :", rowadr[:12])
print("used nnz:", int(rownnz.sum()), "of njmax_nnz", ds.njmax_nnz,
      "| rows per rownnz:", dict(zip(*[v.tolist() for v in np.unique(rownnz, return_counts=True)])))
print("nv_pad rule (nv <= 48): round_up(nv, 4) =", (mjm.nv + 3) // 4 * 4)
for r in (0, int(np.argmin(rownnz))):             # a box-box row and a box-floor row
    a, k = rowadr[r], rownnz[r]
    print(f"row {r:2d} colind {colind[a:a + k]}  (descending)")
    print(f"       vals   {vals[a:a + k]}")

# rebuild the dense matrix from CSR and compare with the forced-dense run (same rows?)
Jcsr = np.zeros((n, mjm.nv), dtype=np.float32)
for r in range(n):
    a, k = rowadr[r], rownnz[r]
    Jcsr[r, colind[a:a + k]] = vals[a:a + k]
Jd = dd.efc.J.numpy()[0, :n, :mjm.nv]
key = lambda J: sorted(tuple(np.round(row, 4)) for row in J)          # row order may differ
print("same set of rows in both layouts:", key(Jcsr) == key(Jd))
print(f"bytes ratio dense / sparse: {results['dense']['nbytes'] / results['auto']['nbytes']:.2f}")
for mode in ("auto", "dense"):
    print(f"overflow ({mode}):",
          "none" if not results[mode]["d"].overflow.numpy().any() else
          results[mode]["d"].overflow.numpy())

if args.dump_json:
    s = n                                         # all rows: the figure draws every one
    out = {"nv": mjm.nv, "nv_pad": int(results["dense"]["m"].nv_pad), "nefc": n,
           "njmax": NJMAX, "njmax_pad": int(dd.njmax_pad), "njmax_nnz": int(ds.njmax_nnz),
           "bytes_sparse": int(results["auto"]["nbytes"]), "bytes_dense":
           int(results["dense"]["nbytes"]), "nworld": NWORLD,
           "contact": ds.efc.id.numpy()[0, :s].tolist(),
           "rownnz": rownnz[:s].tolist(), "rowadr": rowadr[:s].tolist(),
           "colind": colind[:int(rowadr[s - 1] + rownnz[s - 1])].tolist(),
           "vals": vals[:int(rowadr[s - 1] + rownnz[s - 1])].round(3).tolist()}
    with open(args.dump_json, "w") as fh:
        json.dump(out, fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. Softness: impedance and D
`imp(x)` with `x = |pos|/width` is a sigmoid from dmin to dmax built from two power curves joined at `mid`. `D = imp / ((1 - imp) invweight)`.

In [ ]:
# Example 19.4 - ex4_impedance.py
"""Example 19.4 - the impedance curve imp(x) from solimp, and the D it produces, vs d.efc.D.

solimp = (dmin, dmax, width, mid, power) shapes how "hard" a constraint is as it is violated
more deeply. We use a wide transition (width 1 cm) so the curve is visible, press a ball into
the floor by a different depth in each world, and compare D = imp / ((1 - imp) * invweight)
computed in NumPy with d.efc.D. condim="1" on both geoms gives one row per contact.
Run: python ex4_impedance.py [--dump-json PATH] [--plot PNG]
  --dump-json PATH writes the curve and the measured points for the impedance figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

SOLIMP = (0.5, 0.95, 0.01, 0.3, 3.0)          # dmin dmax width mid power
BALL = f"""
<mujoco model="ball_imp">
  <option timestep="0.002"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1" condim="1" solimp="{' '.join(map(str, SOLIMP))}"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" condim="1"
            solimp="{' '.join(map(str, SOLIMP))}"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--plot", default=None)
args, _ = parser.parse_known_args()


def impedance(x, solimp):
    """NumPy copy of the imp branch of constraint.py::_contact_kbimp; x = |pos| / width."""
    dmin, dmax, width, mid, power = solimp
    x = np.clip(np.asarray(x, dtype=float), 0.0, 1.0)
    lo = dmin + (dmax - dmin) * x**power / mid ** (power - 1)
    hi = dmin + (dmax - dmin) * (1 - (1 - x) ** power / (1 - mid) ** (power - 1))
    return np.where(x <= mid, lo, hi)


PEN = np.array([0.0005, 0.001, 0.002, 0.003, 0.005, 0.007, 0.01, 0.015])
NWORLD = len(PEN)

mjm = mujoco.MjModel.from_xml_string(BALL)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=2, njmax=4)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 2] = 0.1 - PEN
d.qpos.assign(qpos)
mjw.forward(m, d)

invweight = float(m.body_invweight0.numpy()[0, 1, 0])
imp_np = impedance(PEN / SOLIMP[2], SOLIMP)
D_np = imp_np / ((1 - imp_np) * invweight)
D_mjw = d.efc.D.numpy()[:, 0]
print(f"solimp {SOLIMP}  invweight0 {invweight:.3f}")
print(" depth[mm]   x=|pos|/width   imp     D numpy   d.efc.D")
for w in range(NWORLD):
    print(f"  {PEN[w]*1e3:6.2f}      {PEN[w]/SOLIMP[2]:6.3f}       {imp_np[w]:.4f}  {D_np[w]:8.3f}"
          f"  {D_mjw[w]:8.3f}")
print(f"max relative difference in D: {np.abs(D_np - D_mjw).max() / D_mjw.max():.1e}")
print("D grows from dmin/(1-dmin) =", round(SOLIMP[0] / (1 - SOLIMP[0]), 3),
      "to dmax/(1-dmax) =", round(SOLIMP[1] / (1 - SOLIMP[1]), 3), "(times 1/invweight)")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

xs = np.linspace(0, 1.2, 61)
curve = impedance(xs, SOLIMP)
if args.dump_json:
    with open(args.dump_json, "w") as fh:
        json.dump({"solimp": SOLIMP, "x": xs.round(4).tolist(), "imp": curve.round(5).tolist(),
                   "pts": [[round(float(p / SOLIMP[2]), 4), round(float(i), 5), round(float(Dm), 4)]
                           for p, i, Dm in zip(PEN, imp_np, D_mjw)]}, fh, separators=(",", ":"))
    print("wrote", args.dump_json)
if args.plot:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        plt.plot(xs, curve, label="imp(x)")
        plt.plot(PEN / SOLIMP[2], D_mjw * invweight / (1 + D_mjw * invweight), "o",
                 label="from d.efc.D")
        plt.xlabel("x = |pos| / width"); plt.ylabel("impedance"); plt.legend()
        plt.savefig(args.plot, dpi=80)
        print("saved", args.plot)
    except ImportError:
        print("skipped: matplotlib not installed")

## 8. A per-world solref sweep
`geom_solref` is batched over worlds (`put_model(..., batch_sizes=...)`): eight time constants in one launch. Larger timeconst, softer contact, deeper penetration. The static model `p = g (1 - imp) / (k imp^2)` predicts the rest penetration.

In [ ]:
# Example 19.5 - ex5_solref_sweep.py
"""Example 19.5 - a per-world solref sweep: the contact's time constant sets how deep a ball sinks.

BALL_DROP (from 0.3 m) with geom_solref batched over worlds: world w uses timeconst
tc_w in [0.005, 0.1] s, dampratio 1. Stiffness k = 1/(dmax^2 tc^2 dr^2) falls as 1/tc^2, so the
ball sinks deeper and rings longer as tc grows. We print the maximum and final penetration and
the static penetration predicted by the soft-constraint model (a* = 0 at rest).
Run: python ex5_solref_sweep.py [--dump-json PATH]
  --dump-json PATH writes the penetration over time per world for the sweep figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 0.3">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 8                                       # one world per time constant
NSTEP = 400                                      # 0.8 s: fall 0.2 m, land, settle
TC = np.geomspace(0.005, 0.1, NWORLD)

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm, batch_sizes={"geom_solref": NWORLD})
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
solref = m.geom_solref.numpy()                   # (NWORLD, ngeom, 2)
solref[:, :, 0] = TC[:, None]                    # floor and ball: same solref, so the mix is TC
solref[:, :, 1] = 1.0
m.geom_solref.assign(solref)
print("geom_solref shape:", m.geom_solref.shape)

pen = np.zeros((NSTEP, NWORLD))
for i in range(NSTEP):
    mjw.step(m, d)
    pen[i] = 0.1 - d.qpos.numpy()[:, 2]          # reads to host each step: demo only
wp.synchronize()

# static penetration: 0 = m a0 + D aref with aref = -k imp pos, D = imp / ((1 - imp) invweight)
dmin, dmax, width, mid, power = mjm.geom_solimp[1]


def imp_of(p):
    x = min(abs(p) / width, 1.0)
    if x <= mid:
        return dmin + (dmax - dmin) * x**power / mid ** (power - 1)
    return dmin + (dmax - dmin) * (1 - (1 - x) ** power / (1 - mid) ** (power - 1))


g = -mjm.opt.gravity[2]
print("  world  timeconst   k [1/s^2]   max pen [mm]  final pen [mm]  static model [mm]")
for w in range(NWORLD):
    tc = max(TC[w], 2 * mjm.opt.timestep)
    k = 1.0 / (dmax**2 * tc**2)
    p = 1e-4
    for _ in range(200):                         # fixed point p = g (1 - imp) / (k imp^2)
        imp = imp_of(p)
        p = 0.5 * p + 0.5 * g * (1 - imp) / (k * imp**2)
    print(f"  {w:5d}   {TC[w]:.4f}   {k:10.1f}    {pen[:, w].max()*1e3:9.3f}     "
          f"{pen[-1, w]*1e3:9.3f}       {p*1e3:9.3f}")
print("max penetration grows with timeconst:", bool(np.all(np.diff(pen.max(axis=0)) > 0)))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    t0 = 80                                      # skip the free fall before landing
    frames = pen[t0::4].round(5) * 1e3           # mm, every 4th step
    with open(args.dump_json, "w") as fh:
        json.dump({"nworld": NWORLD, "dt": 4 * float(mjm.opt.timestep),
                   "t0": t0 * float(mjm.opt.timestep), "tc": TC.round(4).tolist(),
                   "frames": frames.round(3).tolist()}, fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 9. Cones and condim
Pyramidal: `2*(condim-1)` independent rows `n +- mu t` (1 for condim 1). Elliptic: `condim` coupled rows with SATISFIED / CONE / QUADRATIC zones. Each (cone, condim) combination compiles a few kernels the first time.

In [ ]:
# Example 19.6 - ex6_cones_condim.py
"""Example 19.6 - pyramidal vs elliptic cones and condim 1/3/4/6: rows per contact and forces.

A ball rests on the floor while sliding along x, spinning about z and rolling about y, so every
friction direction has something to resist. A contact takes the larger condim and the larger
friction of its two geoms, so both geoms get the same values. For each cone and condim we build
a model, run one forward, and print the rows of the single contact (type, state, efc.force)
and the contact-frame force from mjw.contact_force (normal, two tangents, torsion, two rolling
torques).
cone and condim change kernel specializations, so each new combination compiles a few kernels.
Run: python ex6_cones_condim.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True, linewidth=110)

BALL = """
<mujoco model="ball_cone">
  <option timestep="0.002" cone="{cone}"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1" condim="{condim}" friction="0.8 0.02 0.01"/>
    <body name="ball" pos="0 0 0.0995">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" condim="{condim}"
            friction="0.8 0.02 0.01"/>
    </body>
  </worldbody>
</mujoco>
"""

STATE = {int(v): k[len("mjCNSTRSTATE_"):] for k, v in mujoco.mjtConstraintState.__members__.items()}
TYPE = {int(v): k[len("mjCNSTR_CONTACT_"):] for k, v in mujoco.mjtConstraint.__members__.items()}

print("cone       condim  rows  type          states / efc.force per row")
summary = {}
for cone in ("pyramidal", "elliptic"):
    for condim in (1, 3, 4, 6):
        mjm = mujoco.MjModel.from_xml_string(BALL.format(cone=cone, condim=condim))
        m = mjw.put_model(mjm)
        d = mjw.make_data(mjm, nworld=1, nconmax=2, njmax=16)
        qvel = np.array([[0.5, 0.0, 0.0, 0.0, 1.0, 3.0]])   # slide x, roll about y, spin about z
        d.qvel.assign(qvel)
        mjw.forward(m, d)
        n = int(d.nefc.numpy()[0])
        states = " ".join(STATE[s][:4] for s in d.efc.state.numpy()[0, :n])
        kind = TYPE[int(d.efc.type.numpy()[0, 0])]
        print(f"{cone:10s} {condim:5d} {n:5d}   {kind:12s}  {states}")
        print(f"{'':31s}{d.efc.force.numpy()[0, :n].round(3)}")
        ids = wp.array([0], dtype=int)
        cf = wp.zeros(1, dtype=wp.spatial_vector)
        mjw.contact_force(m, d, ids, False, cf)              # contact frame: normal first
        summary[(cone, condim)] = cf.numpy()[0]
        print(f"{'':31s}efc_address {d.contact.efc_address.numpy()[0]}"
              f"  (nmaxpyramid = {d.contact.efc_address.shape[1]})")

print("\ncontact-frame force [normal, t1, t2 | torsion, roll1, roll2] from mjw.contact_force:")
for (cone, condim), f in summary.items():
    print(f"  {cone:10s} condim {condim}: {f.round(3)}")
print("rows per contact: pyramidal 2*(condim-1) (1 for condim 1), elliptic condim")

## 9. Friction loss: a Huber row
Quadratic for `|x| < frictionloss/D` (the joint sticks with a small soft creep), linear beyond (LINEARPOS/LINEARNEG, force exactly -+frictionloss).

In [ ]:
# Example 19.7 - ex7_frictionloss.py
"""Example 19.7 - joint frictionloss: a Huber cost that sticks, then saturates and slides.

A wheel on a vertical hinge (gravity does no work on it) has frictionloss 0.5 N m. World w
pushes it with a motor torque tau_w from 0 to 1 N m. Below the frictionloss the friction row
stays QUADRATIC (the wheel sticks, with a small soft creep); above it the row saturates in the
LINEARPOS state with force exactly -frictionloss, and the wheel spins up.
Run: python ex7_frictionloss.py [--dump-json PATH]
  --dump-json PATH writes (torque, force, state) per world for the Huber figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

WHEEL = """
<mujoco model="wheel">
  <option timestep="0.002"/>
  <worldbody>
    <body name="wheel" pos="0 0 0.5">
      <joint name="axle" type="hinge" axis="0 0 1" frictionloss="0.5"/>
      <geom type="cylinder" size=".2 .02" mass="2"/>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="axle" gear="1"/></actuator>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

STATE = {int(v): k[len("mjCNSTRSTATE_"):] for k, v in mujoco.mjtConstraintState.__members__.items()}
NWORLD = 12
NSTEP = 50
TAU = np.linspace(0.0, 1.0, NWORLD - 1).tolist() + [0.49]     # plus one just below 0.5

mjm = mujoco.MjModel.from_xml_string(WHEEL)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, njmax=4)
d.ctrl.assign(np.array(TAU, dtype=np.float32)[:, None])
mjw.forward(m, d)

# .copy(): on the CPU device .numpy() is a view of the Warp buffer; the steps below overwrite it
st = d.efc.state.numpy()[:, 0].copy()
force = d.efc.force.numpy()[:, 0].copy()
qacc = d.qacc.numpy()[:, 0].copy()
D = float(d.efc.D.numpy()[0, 0])
print(f"nf {d.nf.numpy()[0]}  efc.type {mujoco.mjtConstraint(int(d.efc.type.numpy()[0, 0])).name}"
      f"  D {D:.3f}  frictionloss {mjm.dof_frictionloss[0]}  rf = frictionloss / D = "
      f"{mjm.dof_frictionloss[0] / D:.4f}")
for _ in range(NSTEP):
    mjw.step(m, d)
qvel = d.qvel.numpy()[:, 0]
force_late = d.efc.force.numpy()[:, 0]
print(" world  torque   state       efc.force   qacc     | after 0.1 s: qvel    efc.force")
order = np.argsort(TAU)
prev = None
for w in order:
    tag = "   <- transition" if prev is not None and st[w] != prev else ""
    print(f"  {w:3d}   {TAU[w]:5.2f}    {STATE[st[w]]:10s} {force[w]:8.4f}  "
          f"{qacc[w]:8.4f}  |      {qvel[w]:8.4f}  {force_late[w]:8.4f}{tag}")
    prev = st[w]
print("saturated rows carry exactly -frictionloss:",
      bool(np.allclose(force[st == 3], -mjm.dof_frictionloss[0])))

if args.dump_json:
    with open(args.dump_json, "w") as fh:
        json.dump({"frictionloss": float(mjm.dof_frictionloss[0]), "D": round(D, 5),
                   "pts": [[round(float(TAU[w]), 4), round(float(force[w]), 5), STATE[st[w]]]
                           for w in order]}, fh, separators=(",", ":"))
    print("wrote", args.dump_json)

## 10. Equalities
CONNECT (3 rows), WELD (6), JOINT (1), TENDON (1). Rows come out per kernel type (connect, weld, joint, tendon), not in eq-id order; map rows with `efc.id`. FLEX and FLEXSTRAIN need a flex and are shown in Ch 23.

In [ ]:
# Example 19.8 - ex8_equalities.py
"""Example 19.8 - four equality types in one model: rows per equality, row order and violations.

The equalities are declared in the order TENDON, JOINT, WELD, CONNECT (eq ids 0..3):
  TENDON : two slider tendons must have equal length (1 row)
  JOINT  : hinge b = 0.5 * hinge a (1 row, polycoef)
  WELD   : a free box is welded to the world frame where it starts (6 rows)
  CONNECT: a free capsule hangs from a world point, like a ball joint (3 rows)
make_constraint launches one kernel per type (connect, weld, joint, tendon), so the rows come out
grouped by type, not by eq id. Gravity pulls on everything; each equality has its own eq_solref,
and the soft rows show a small steady violation efc.pos that depends on the load each equality
carries as well as on its solref (an unloaded equality, like the tendon here, ends at zero).
FLEX and FLEXSTRAIN rows need a flex; they are shown in Ch 23.
Run: python ex8_equalities.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True, linewidth=100)

EQUALITIES = """
<mujoco model="equalities">
  <option timestep="0.002"/>
  <worldbody>
    <body name="sl1" pos="-1 0 1">
      <joint name="s1" type="slide" axis="0 0 1"/>
      <geom type="box" size=".05 .05 .05" mass="1" contype="0" conaffinity="0"/>
    </body>
    <body name="sl2" pos="-0.7 0 1">
      <joint name="s2" type="slide" axis="0 0 1"/>
      <geom type="box" size=".05 .05 .05" mass="2" contype="0" conaffinity="0"/>
    </body>
    <body name="arm_a" pos="0 0 1">
      <joint name="ha" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".02" mass="1" contype="0" conaffinity="0"/>
    </body>
    <body name="arm_b" pos="0 0.3 1">
      <joint name="hb" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".02" mass="1" contype="0" conaffinity="0"/>
    </body>
    <body name="welded" pos="0.7 0 1">
      <freejoint/>
      <geom type="box" size=".05 .05 .05" mass="1" contype="0" conaffinity="0"/>
    </body>
    <body name="hanging" pos="1.2 0 1">
      <freejoint/>
      <geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02" mass="1" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <tendon>
    <fixed name="t1"><joint joint="s1" coef="1"/></fixed>
    <fixed name="t2"><joint joint="s2" coef="1"/></fixed>
  </tendon>
  <equality>
    <tendon tendon1="t1" tendon2="t2" solref="0.02 1"/>
    <joint joint1="hb" joint2="ha" polycoef="0 0.5 0 0 0" solref="0.01 1"/>
    <weld body1="welded" solref="0.005 1"/>
    <connect body1="hanging" anchor="0 0 0" solref="0.04 1"/>
  </equality>
</mujoco>
"""

NWORLD = 2
NSTEP = 200

mjm = mujoco.MjModel.from_xml_string(EQUALITIES)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, njmax=16)
print("eq_type :", [mujoco.mjtEq(int(t)).name for t in mjm.eq_type])
print("eq_solref timeconst:", mjm.eq_solref[:, 0])

qpos = np.tile(mjm.qpos0, (NWORLD, 1))
ha = mjm.jnt_qposadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_JOINT, "ha")]
hb = mjm.jnt_qposadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_JOINT, "hb")]
qpos[:, ha] = 0.4                                 # start the joint equality violated
qpos[:, 0] = 0.05                                 # and the tendon equality (s1 - s2 = 5 cm)
d.qpos.assign(qpos)
q = d.qpos.numpy()[0]
print(f"start: hb - 0.5*ha = {q[hb] - 0.5 * q[ha]:+.2e}, s1 - s2 = {q[0] - q[1]:+.2e}")
for _ in range(NSTEP):
    mjw.step(m, d)

n = int(d.nefc.numpy()[0])
eqid = d.efc.id.numpy()[0, :n]
pos = d.efc.pos.numpy()[0, :n]
print(f"ne {d.ne.numpy()}  nefc {d.nefc.numpy()}  (3 + 6 + 1 + 1 = 11)")
print("row -> eq id:", eqid)
for e in range(mjm.neq):
    rows = np.flatnonzero(eqid == e)
    print(f"  eq {e} {mujoco.mjtEq(int(mjm.eq_type[e])).name[5:]:8s} rows {rows.tolist()!s:22s}"
          f" max |pos| = {np.abs(pos[rows]).max():.2e}")
q = d.qpos.numpy()[0]
print(f"after {NSTEP * mjm.opt.timestep:.1f} s: hb - 0.5*ha = {q[hb] - 0.5 * q[ha]:+.2e},"
      f" s1 - s2 = {q[0] - q[1]:+.2e}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## Exercise 19.1: predict nefc
Write `expected_rows(ncon, condim, cone)` and assert it against `d.nefc` for both cones and condim 1, 3, 4, 6. The scene has 1 CONNECT equality (3 rows), 1 friction-loss dof, 1 violated hinge limit and 4 box-floor contacts. A contact takes the larger condim of its two geoms, so the floor gets the same condim.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SCENE = """
<mujoco><option timestep="0.005" cone="{cone}"/><worldbody>
  <geom type="plane" size="5 5 .1" condim="{condim}"/>
  <body pos="0 0 0.099"><freejoint/><geom type="box" size=".1 .1 .1" condim="{condim}"/></body>
  <body pos="1 0 1"><joint name="h" type="hinge" axis="0 1 0" range="-30 30" frictionloss="0.1"/>
    <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".03" contype="0" conaffinity="0"/></body>
  <body name="hang" pos="2 0 1"><freejoint/>
    <geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02" contype="0" conaffinity="0"/></body>
</worldbody>
<equality><connect body1="hang" anchor="0 0 0"/></equality></mujoco>"""


def expected_rows(ncon, condim, cone, ne=3, nf=1, nl=1):
    ...  # your code: rows for ne + nf + nl + ncon contacts



for cone in ("pyramidal", "elliptic"):
    for condim in (1, 3, 4, 6):
        mjm = mujoco.MjModel.from_xml_string(SCENE.format(cone=cone, condim=condim))
        mjd = mujoco.MjData(mjm)
        mjd.qpos[7] = np.radians(40)          # the hinge starts past its +30 degree limit
        m = mjw.put_model(mjm)
        d = mjw.put_data(mjm, mjd, nworld=1, nconmax=8, njmax=64)
        mjw.forward(m, d)
        got = int(d.nefc.numpy()[0])
        assert got == expected_rows(4, condim, cone), (cone, condim, got)
print("Exercise 19.1 passed")

In [ ]:
#@title Exercise 19.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

SCENE = """
<mujoco><option timestep="0.005" cone="{cone}"/><worldbody>
  <geom type="plane" size="5 5 .1" condim="{condim}"/>
  <body pos="0 0 0.099"><freejoint/><geom type="box" size=".1 .1 .1" condim="{condim}"/></body>
  <body pos="1 0 1"><joint name="h" type="hinge" axis="0 1 0" range="-30 30" frictionloss="0.1"/>
    <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".03" contype="0" conaffinity="0"/></body>
  <body name="hang" pos="2 0 1"><freejoint/>
    <geom type="capsule" fromto="0 0 0 0 0 -.3" size=".02" contype="0" conaffinity="0"/></body>
</worldbody>
<equality><connect body1="hang" anchor="0 0 0"/></equality></mujoco>"""


def expected_rows(ncon, condim, cone, ne=3, nf=1, nl=1):
    if cone == "elliptic":
        per = condim
    else:
        per = 1 if condim == 1 else 2 * (condim - 1)
    return ne + nf + nl + ncon * per


for cone in ("pyramidal", "elliptic"):
    for condim in (1, 3, 4, 6):
        mjm = mujoco.MjModel.from_xml_string(SCENE.format(cone=cone, condim=condim))
        mjd = mujoco.MjData(mjm)
        mjd.qpos[7] = np.radians(40)          # the hinge starts past its +30 degree limit
        m = mjw.put_model(mjm)
        d = mjw.put_data(mjm, mjd, nworld=1, nconmax=8, njmax=64)
        mjw.forward(m, d)
        got = int(d.nefc.numpy()[0])
        assert got == expected_rows(4, condim, cone), (cone, condim, got)
print("Exercise 19.1 passed")

## Exercise 19.2: force only on active pyramid rows
On the ex3 scene, assert from MJWarp's arrays that every pyramidal row force is >= 0, zero where `J qacc - aref >= 0`, and positive where it is clearly negative. Return the active-row count per world.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TILTED_BOX = """
<mujoco><option timestep="0.002" cone="pyramidal"/><worldbody>
  <geom type="plane" size="5 5 .1"/>
  <body pos="0 0 .098" euler="0 10 0"><freejoint/><geom type="box" size=".1 .1 .1" friction="0.5"/></body>
</worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(TILTED_BOX)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=8, njmax=32)
d.qvel.assign(np.array([[v, 0, 0, 0, 0, 0] for v in (0.0, 0.3, 1.0, 3.0)]))
mjw.forward(m, d)


def check_world(w):
    ...  # your code: assert the force/activity rule, return the number of active rows



counts = [check_world(w) for w in range(d.nworld)]
print("active rows per world:", counts)
assert counts == [8, 6, 6, 2]
print("Exercise 19.2 passed")

In [ ]:
#@title Exercise 19.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

TILTED_BOX = """
<mujoco><option timestep="0.002" cone="pyramidal"/><worldbody>
  <geom type="plane" size="5 5 .1"/>
  <body pos="0 0 .098" euler="0 10 0"><freejoint/><geom type="box" size=".1 .1 .1" friction="0.5"/></body>
</worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(TILTED_BOX)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=8, njmax=32)
d.qvel.assign(np.array([[v, 0, 0, 0, 0, 0] for v in (0.0, 0.3, 1.0, 3.0)]))
mjw.forward(m, d)


def check_world(w):
    n = int(d.nefc.numpy()[w])
    J = d.efc.J.numpy()[w, :n, :mjm.nv].astype(np.float64)
    aref = d.efc.aref.numpy()[w, :n]
    f = d.efc.force.numpy()[w, :n]
    jar = J @ d.qacc.numpy()[w] - aref
    active = jar < 0
    tol = 1e-3 * max(1.0, np.abs(f).max())
    assert np.all(f >= -tol)
    assert np.all(f[~active] <= tol)
    assert np.all(f[active & (jar < -1e-3)] > 0)
    return int(active.sum())


counts = [check_world(w) for w in range(d.nworld)]
print("active rows per world:", counts)
assert counts == [8, 6, 6, 2]
print("Exercise 19.2 passed")

## Stretch: the elliptic cone in NumPy
Switch the ex3 scene to `cone="elliptic"` (3 rows per contact). Group rows by `efc.id`; per contact compute `N = mu x0` and `T = ||(mu_i x_i)||` from the residuals `x = J qacc - aref`; pick the zone as in `solver.py::_eval_constraint` (top: 0; bottom: sum 0.5 D_i x_i^2; middle: 0.5 dm (N - mu T)^2 with dm = D0 / (mu^2 (1 + mu^2))); check that random perturbations of qacc never lower your cost and that your forces match `efc.force`.